# Main

In [20]:
import helpers.assembly as assembly
import helpers.element as element
import helpers.interfaces as interfaces
import helpers.partition as partition
import helpers.preprocess as preprocess
import numpy as np

In [21]:
nodes = {
    1:  (0.0,    0.0, 0.0),      # B0
    2:  (4.744,  0.0, 0.0),      # B1
    3:  (4.744,  0.0, 6.796),   # T1

    4:  (9.488,  0.0, 0.0),      # B2
    5:  (9.488,  0.0, 7.724),   # T2

    6:  (14.232, 0.0, 0.0),      # B3
    7:  (14.232, 0.0, 8.419),   # T3

    8:  (18.976, 0.0, 0.0),      # B4
    9:  (18.976, 0.0, 8.883),   # T4

    10: (23.720, 0.0, 0.0),      # B5
    11: (23.720, 0.0, 9.115),   # T5

    12: (28.464, 0.0, 0.0),      # B6
    13: (28.464, 0.0, 9.115),   # T6

    14: (33.208, 0.0, 0.0),      # B7
    15: (33.208, 0.0, 8.883),   # T7

    16: (37.952, 0.0, 0.0),      # B8
    17: (37.952, 0.0, 8.419),   # T8

    18: (42.696, 0.0, 0.0),      # B9
    19: (42.696, 0.0, 7.724),   # T9

    20: (47.440, 0.0, 0.0),      # B10
    21: (47.440, 0.0, 6.796),   # T10

    22: (52.184, 0.0, 0.0),      # B11

    23:  (0.0,    10.0, 0.0),      # B0'
    24:  (4.744,  10.0, 0.0),      # B1'
    25:  (4.744,  10.0, 6.796),   # T1'

    26:  (9.488,  10.0, 0.0),      # B2'
    27:  (9.488,  10.0, 7.724),   # T2'

    28:  (14.232, 10.0, 0.0),      # B3'
    29:  (14.232, 10.0, 8.419),   # T3'

    30:  (18.976, 10.0, 0.0),      # B4'
    31:  (18.976, 10.0, 8.883),   # T4'

    32: (23.720, 10.0, 0.0),      # B5'
    33: (23.720, 10.0, 9.115),   # T5'

    34: (28.464, 10.0, 0.0),      # B6'
    35: (28.464, 10.0, 9.115),   # T6'

    36: (33.208, 10.0, 0.0),      # B7'
    37: (33.208, 10.0, 8.883),   # T7'

    38: (37.952, 10.0, 0.0),      # B8'
    39: (37.952, 10.0, 8.419),   # T8'

    40: (42.696, 10.0, 0.0),      # B9'
    41: (42.696, 10.0, 7.724),   # T9'

    42: (47.440, 10.0, 0.0),      # B10'
    43: (47.440, 10.0, 6.796),   # T10'

    44: (52.184, 10.0, 0.0),      # B11'
}

In [22]:
filepath = "inputs/validation_truss.json"

nodes = interfaces.read_nodes(filepath)
elements = interfaces.read_elements(filepath)
constraints = interfaces.read_constraints(filepath)
materials = interfaces.read_materials(filepath)
sections = interfaces.read_sections(filepath)
constraints = interfaces.read_constraints(filepath)
nodal_loads = interfaces.read_nodal_loads(filepath)
element_loads = interfaces.read_element_loads(filepath)
temperature_loads = interfaces.read_temperature_loads(filepath)
fabrication_loads = interfaces.read_fabrication_errors(filepath)

print(element_loads)


{1: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], 2: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]}


In [23]:
k_list = []
T_list = []
Qf_list = []
Qh_list = []
Qe_list = []
map_list = []

element_paras = element.build_elements_para(nodes, elements, materials, sections)

for e_id in elements:
        L, etype, E, G, A, Iy, Iz, J, mt, alpha = element_paras[e_id]
        kl = element.element_kl(E, G, A, Iy, Iz, J, L)
        gamma = element.build_gamma_3d(nodes[elements[e_id][3][0]], nodes[elements[e_id][3][1]], psi=0.0, angle_unit="deg")
        T = element.frame_transformation_matrix(gamma)
        Qf = preprocess.fef_cal(element_loads.get(e_id), L, angle_unit="deg")
        Qh = preprocess.heat_cal(temperature_loads.get(e_id), E, A, alpha)
        Qe = preprocess.fabrication_error_cal(fabrication_loads.get(e_id), E, A, L)
        m = element.element_dof_map_1based(elements[e_id][3][0], elements[e_id][3][1])


        k_list.append(kl)
        T_list.append(T)
        Qf_list.append(Qf)
        Qh_list.append(Qh)
        Qe_list.append(Qe)
        map_list.append(m)

Qt_list = [x + y + z for x, y, z in zip(Qf_list, Qh_list, Qe_list)]

ndof = int(np.max(np.concatenate(map_list)))


In [24]:
dof_restrained_1based = element.restrained_dofs_1based(constraints, element.node_dofs_1based)
dof_fictitious_1based = np.array([], dtype=int)

dof_restrained_1based = np.sort(
    np.concatenate((dof_restrained_1based, dof_fictitious_1based))
)

F_global = np.zeros(ndof, dtype=float)
u_global = np.zeros(ndof, dtype=float)

In [25]:
K_global, F_fef_global = assembly.assemble_global_stiffness_and_fef(
    ndof,
    k_list,
    T_list,
    Qf_list,
    map_list
    )
print(K_global,"\n")
print(F_fef_global)

[[5011.70668946    0.         1130.62215725 ...    0.
     0.            0.        ]
 [   0.         2026.69461731    0.         ...    0.
     0.            0.        ]
 [1130.62215725    0.         1648.76328947 ...    0.
     0.            0.        ]
 ...
 [   0.            0.            0.         ...  164.15578668
     0.           40.94787341]
 [   0.            0.            0.         ...    0.
   272.85132216    0.        ]
 [   0.            0.            0.         ...   40.94787341
     0.          286.49928687]] 

[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 

In [26]:
(
    K_ff,
    K_fr,
    K_rf,
    K_rr,
    f_f,
    f_r,
    u_r,
    f_fef_f,
    f_fef_r,
    free_dofs,
    restrained_dofs,
) = partition.partition_system(
    K_global,
    F_global,
    u_global,
    F_fef_global,
    dof_restrained_1based
)


In [27]:
rhs = f_f - f_fef_f - K_fr @ u_r
u_f = np.linalg.solve(K_ff, rhs)

F_r = K_rf @ u_f + K_rr @ u_r + f_fef_r

u_global = assembly.assemble_global_displacements(
    u_f,
    u_r,
    free_dofs,
    restrained_dofs
    )

f_global_complete = assembly.assemble_global_forces(
    f_f,
    F_r,
    free_dofs,
    restrained_dofs
    )